# Startup Team Matching Platform
## NLP-Based Skill Matching | FastAPI Backend | Recommendation Engine

Match cofounder candidates with startup ideas using semantic similarity.

---

In [ ]:
# CELL 1: Setup
import subprocess, sys
for p in ['numpy','pandas','scikit-learn','sentence-transformers','fastapi','uvicorn','sqlite3']:
    try: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
    except: pass

import numpy as np, pandas as pd, json, sqlite3
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
np.random.seed(42)
print('Platform Ready!')

In [ ]:
# CELL 2: Database Schema & Sample Data
DB_PATH = OUTPUT_DIR / 'startup_platform.db'
conn = sqlite3.connect(str(DB_PATH))
cursor = conn.cursor()

# Create tables
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL, email TEXT UNIQUE,
        skills TEXT, interests TEXT, bio TEXT,
        looking_for TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS startups (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL, description TEXT,
        industry TEXT, stage TEXT, skills_needed TEXT,
        founder_id INTEGER, FOREIGN KEY(founder_id) REFERENCES users(id)
    );
    CREATE TABLE IF NOT EXISTS matches (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER, startup_id INTEGER,
        score REAL, status TEXT DEFAULT 'pending',
        FOREIGN KEY(user_id) REFERENCES users(id),
        FOREIGN KEY(startup_id) REFERENCES startups(id)
    );
''')

# Insert sample data
users = [
    ('Alice Chen', 'alice@test.com', 'machine learning,python,data science', 'healthcare AI', 'ML engineer with 3y experience', 'cofounder'),
    ('Bob Kumar', 'bob@test.com', 'fullstack,react,nodejs,aws', 'edtech,saas', 'Senior dev, built 2 startups', 'technical cofounder'),
    ('Carol Wu', 'carol@test.com', 'product management,growth,marketing', 'fintech,crypto', 'Ex-PM at top tech company', 'founding team'),
    ('Dave Park', 'dave@test.com', 'deep learning,computer vision,pytorch', 'robotics,autonomous', 'PhD in CV, published researcher', 'cofounder'),
    ('Eva Schmidt', 'eva@test.com', 'ui/ux,design,figma,branding', 'consumer apps', 'Lead designer, design thinking expert', 'design cofounder'),
]
startups = [
    ('MedVision AI', 'AI-powered medical imaging diagnostics platform', 'healthcare', 'idea', 'deep learning,medical,python'),
    ('EduFlow', 'Personalized adaptive learning platform for K-12', 'edtech', 'mvp', 'fullstack,react,ml'),
    ('GreenChain', 'Blockchain-based carbon credit marketplace', 'climate tech', 'seed', 'blockchain,fullstack,product'),
    ('AutoDrive', 'Autonomous delivery robot fleet management', 'robotics', 'idea', 'computer vision,robotics,python'),
]

for u in users:
    try: cursor.execute('INSERT INTO users (name,email,skills,interests,bio,looking_for) VALUES (?,?,?,?,?,?)', u)
    except: pass
for s in startups:
    try: cursor.execute('INSERT INTO startups (name,description,industry,stage,skills_needed) VALUES (?,?,?,?,?)', s)
    except: pass
conn.commit()
print(f"Database created: {DB_PATH}")
print(f"Users: {cursor.execute('SELECT COUNT(*) FROM users').fetchone()[0]}")
print(f"Startups: {cursor.execute('SELECT COUNT(*) FROM startups').fetchone()[0]}")

In [ ]:
# CELL 3: NLP-Based Matching Engine
class MatchingEngine:
    """Semantic similarity matching using TF-IDF + cosine similarity."""
    
    def __init__(self, db_path):
        self.conn = sqlite3.connect(str(db_path))
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
    
    def _build_user_profile(self, user):
        return f"{user[3]} {user[4]} {user[5]}"  # skills, interests, bio
    
    def _build_startup_profile(self, startup):
        return f"{startup[1]} {startup[2]} {startup[3]} {startup[4]}"
    
    def find_matches(self, top_k=3):
        """Find top-K startup matches for each user."""
        users = self.conn.execute('SELECT * FROM users').fetchall()
        startups = self.conn.execute('SELECT * FROM startups').fetchall()
        
        user_texts = [self._build_user_profile(u) for u in users]
        startup_texts = [self._build_startup_profile(s) for s in startups]
        
        all_texts = user_texts + startup_texts
        tfidf = self.vectorizer.fit_transform(all_texts)
        
        user_vecs = tfidf[:len(users)]
        startup_vecs = tfidf[len(users):]
        
        sim_matrix = cosine_similarity(user_vecs, startup_vecs)
        
        results = []
        for i, user in enumerate(users):
            scores = sim_matrix[i]
            top_indices = scores.argsort()[::-1][:top_k]
            for idx in top_indices:
                results.append({
                    'user': user[1], 'startup': startups[idx][1],
                    'score': round(scores[idx], 4),
                    'industry': startups[idx][3]
                })
        return pd.DataFrame(results)

engine = MatchingEngine(DB_PATH)
matches = engine.find_matches(top_k=2)
print("Top Matches:")
print(matches.to_string(index=False))
matches.to_csv(OUTPUT_DIR / 'matches.csv', index=False)
conn.close()
print("\nP7 Startup Team Matching - COMPLETE!")